# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q imbalanced-learn faker pandas seaborn matplotlib scikit-learn
    print('Packages installed.')


# Handling Imbalanced Datasets (SMOTE & ADASYN) In-Depth

In social science, interesting events are often rare: protests, fraud, hate speech, or disease outbreaks. Standard machine learning models bias towards the majority class, often ignoring the minority class entirely.

This notebook demonstrates how to solve this using **Resampling Techniques** from the `imbalanced-learn` library.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE, ADASYN
from collections import Counter

# Plot settings
sns.set_style("whitegrid")

## 1. Generating Imbalanced Data
We simulate a dataset where only 5% of cases are "positive" (Class 1).

In [ ]:
X, y = make_classification(
    n_samples=1000, 
    n_features=2, 
    n_redundant=0,
    n_clusters_per_class=1, 
    weights=[0.95], # 95% Class 0, 5% Class 1
    flip_y=0, 
    random_state=42
)

print(f"Original Class Distribution: {Counter(y)}")

# Viz
plt.figure(figsize=(8, 6))
sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y, style=y, palette='viridis', alpha=0.7)
plt.title("Original Imbalanced Dataset")
plt.show()

## 2. The Baseline Problem
Training a standard Logistic Regression on this data often yields high accuracy but 0 recall for the minority class.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Baseline Model Report:")
print(classification_report(y_test, y_pred))

## 3. Applying SMOTE
**SMOTE (Synthetic Minority Over-sampling Technique)** creates synthetic examples by interpolating between existing minority instances.

In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print(f"Resampled Class Distribution: {Counter(y_resampled)}")

# Viz Resampled Data
plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_resampled[:, 0], y=X_resampled[:, 1], hue=y_resampled, style=y_resampled, palette='viridis', alpha=0.5)
plt.title("SMOTE Augmented Dataset")
plt.show()

## 4. Retraining on SMOTE Data
Now we train the same model on the balanced dataset.

In [ ]:
model_smote = LogisticRegression()
model_smote.fit(X_resampled, y_resampled)
y_pred_smote = model_smote.predict(X_test)

print("SMOTE Model Report:")
print(classification_report(y_test, y_pred_smote))

## Conclusion
By using SMOTE, we likely sacrificed some precision (more False Positives) to gain significant recall (fewer False Negatives). In social science contexts like fraud detection or crisis response, catching the minority cases is often more important.